# K-Means Clustering vs Batch Correction: ARI Evaluation

Evaluate the effect of fedRBE batch correction on k-means clustering
performance using Adjusted Rand Index (ARI).

**Approach:**
- Load uncorrected (before) and corrected (after) expression matrices from \`evaluation_data/\`
- Run central k-means for two targets: **condition** (biological signal) and **batch** (technical signal)
- Compute ARI between cluster assignments and true labels
- Compare with pre-computed federated k-means results when available

**Expected outcome after successful batch correction:**
- **Condition ARI increases** — biological groups become more separable
- **Batch ARI decreases** — batch structure is removed


In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score

## Configuration

In [ ]:
# Datasets to evaluate. Available: "proteomics", "microarray".
# New datasets work automatically when added to dataset_configs() in
# run_real_datasets_kmeans.py (Python) or rd_dataset_configs() in
# evaluation_utils/real_datasets_utils.R.
DATASETS = ["proteomics", "microarray"]

# Which corrected matrix to use:
#   "auto"      — try federated first, fall back to central
#   "central"   — use centrally corrected matrix only
#   "federated" — use federated corrected matrix only
CORRECTED_SOURCE = "auto"

# Drop feature rows containing NA before k-means
# (required for proteomics; microarray usually has no NAs)
REMOVE_NA = True

# Random seed for k-means reproducibility
SEED = 11

# Number of k-means random initializations (matches run_real_datasets_kmeans.py)
N_INIT = 50

## Setup

In [ ]:

# Resolve paths relative to this notebook.
# REPO_ROOT is used by dataset_configs() to locate evaluation_data/.
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = (NOTEBOOK_DIR / "..").resolve()
REAL_DATASETS_DIR = NOTEBOOK_DIR / "real_datasets"

# Make the helper scripts importable
for p in [str(REPO_ROOT), str(REAL_DATASETS_DIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)

# Import existing data-loading and metric functions from run_real_datasets_kmeans.py.
# This avoids duplicating data parsing, alignment, and scaling logic.
from run_real_datasets_kmeans import (
    dataset_configs,
    discover_clients,
    load_metadata,
    load_feature_matrix,
    load_before_matrix_from_sites,
    align_matrix_to_metadata,
    drop_rows_with_any_na,
    scale_like_federated,
    run_central_kmeans,
    merge_central_clusters,
    align_predictions_to_truth,
    calculate_metrics,
    choose_corrected_path,
)

CONFIGS = dataset_configs(REPO_ROOT)

sns.set_style("whitegrid")
print(f"Available datasets: {sorted(CONFIGS.keys())}")
print(f"Selected: {DATASETS}")

## Helper Functions

In [ ]:
def _is_git_lfs_pointer(path: "Path") -> bool:
    """Check whether a file is an unfetched Git LFS pointer.

    Git LFS pointers are small text files (< 200 bytes) whose first line
    is ``version https://git-lfs``.
    """
    if not path.exists() or path.stat().st_size > 200:
        return False
    try:
        header = path.read_text(errors="replace")[:50]
        return header.startswith("version https://git-lfs")
    except Exception:
        return False


def _load_corrected_with_lfs_fallback(
    cfg, corrected_source: str, clients, metadata
) -> tuple:
    """Load the corrected expression matrix, with Git LFS fallback.

    If the chosen merged corrected file is an unfetched LFS pointer,
    falls back to merging the numbered per-site files
    ``only_batch_corrected_data_N.csv`` from ``individual_results/``.
    These files are matched to sites by overlapping sample columns
    (not by index, since numbering may differ from sorted site order).

    Parameters
    ----------
    cfg : DatasetConfig
        Dataset configuration from ``dataset_configs()``.
    corrected_source : str
        ``"auto"``, ``"central"``, or ``"federated"``.
    clients : list[Path]
        Discovered client directories.
    metadata : pd.DataFrame
        Metadata with ``file`` and ``lab`` columns.

    Returns
    -------
    matrix : pd.DataFrame
        Features x samples expression matrix.
    kind : str
        Human-readable label of the data source used.
    """
    chosen_path, kind = choose_corrected_path(cfg, corrected_source)

    if not _is_git_lfs_pointer(chosen_path):
        return load_feature_matrix(chosen_path), kind

    # --- LFS fallback: merge numbered per-site corrected files ---
    print(f"  [LFS fallback] {chosen_path.name} is an unfetched LFS pointer.")
    after_dir = chosen_path.parent / "individual_results"
    numbered_files = sorted(after_dir.glob("only_batch_corrected_data_*.csv"))

    if not numbered_files:
        raise FileNotFoundError(
            f"LFS fallback failed: no only_batch_corrected_data_*.csv in {after_dir}.\n"
            f"Run 'git lfs pull' to fetch {chosen_path.name}."
        )

    # Merge all numbered files — order doesn't matter since we align by column name
    parts = [load_feature_matrix(f) for f in numbered_files]
    merged = pd.concat(parts, axis=1, join="outer")

    dup_cols = merged.columns[merged.columns.duplicated()].unique().tolist()
    if dup_cols:
        raise ValueError(f"Duplicate sample columns in LFS fallback merge: {dup_cols[:5]}")

    print(
        f"  [LFS fallback] Merged {len(numbered_files)} files from individual_results/ "
        f"({merged.shape[0]} features x {merged.shape[1]} samples)"
    )
    return merged, f"{kind} (LFS fallback)"


def load_dataset(
    dataset_name: str,
    corrected_source: str = "auto",
    remove_na: bool = True,
) -> dict:
    """Load before/after expression matrices and metadata for one dataset.

    Reads data directly from ``evaluation_data/`` — nothing is copied into
    the clustering evaluation folder.

    Parameters
    ----------
    dataset_name : str
        Key in CONFIGS (e.g. ``"proteomics"``, ``"microarray"``).
    corrected_source : str
        ``"auto"`` (try federated, fall back to central), ``"central"``,
        or ``"federated"``.
    remove_na : bool
        If True, drop feature rows containing any NA (needed for k-means).

    Returns
    -------
    dict with keys:
        name, metadata, before_matrix, corrected_matrix, k_condition, k_batch.
    """
    cfg = CONFIGS[dataset_name]
    clients = discover_clients(cfg)
    metadata = load_metadata(cfg, clients)

    # Load uncorrected (before) matrix by merging per-site files
    before_matrix = align_matrix_to_metadata(
        load_before_matrix_from_sites(cfg, clients),
        metadata,
        f"{dataset_name} before",
    )

    # Load corrected matrix (with LFS pointer fallback)
    corrected_raw, corrected_kind = _load_corrected_with_lfs_fallback(
        cfg, corrected_source, clients, metadata
    )
    corrected_matrix = align_matrix_to_metadata(
        corrected_raw, metadata, f"{dataset_name} corrected"
    )

    if remove_na:
        before_matrix = drop_rows_with_any_na(before_matrix, f"{dataset_name} before")
        corrected_matrix = drop_rows_with_any_na(corrected_matrix, f"{dataset_name} corrected")

    k_condition = int(metadata["condition"].nunique())
    k_batch = int(metadata["lab"].nunique())

    print(f"[{dataset_name}] corrected source: {corrected_kind}")
    print(
        f"  samples={metadata.shape[0]}, features_before={before_matrix.shape[0]}, "
        f"features_corrected={corrected_matrix.shape[0]}"
    )
    print(f"  conditions={k_condition}, batches={k_batch}")

    return {
        "name": dataset_name,
        "metadata": metadata,
        "before_matrix": before_matrix,
        "corrected_matrix": corrected_matrix,
        "k_condition": k_condition,
        "k_batch": k_batch,
    }


def evaluate_dataset(ds: dict, seed: int = 11) -> pd.DataFrame:
    """Run central k-means for each (target, k) pair and compute ARI/MCC/accuracy.

    Runs k-means on both the uncorrected and corrected matrices. For each
    target (condition or batch), the cluster labels are aligned to the true
    labels via best-match permutation before computing metrics.

    Parameters
    ----------
    ds : dict
        Output of ``load_dataset()``.
    seed : int
        Random seed for k-means reproducibility.

    Returns
    -------
    pd.DataFrame
        Columns: Dataset, Target, K, Method, ARI, MCC, Accuracy, N.
    """
    k_values = sorted({ds["k_condition"], ds["k_batch"]})

    before_clusters = run_central_kmeans(ds["before_matrix"], k_values, seed)
    corrected_clusters = run_central_kmeans(ds["corrected_matrix"], k_values, seed)
    central_meta = merge_central_clusters(ds["metadata"], before_clusters, corrected_clusters)

    records = []
    for target, k in [("condition", ds["k_condition"]), ("batch", ds["k_batch"])]:
        truth_col = "condition" if target == "condition" else "lab"
        truth = central_meta[truth_col]

        for variant, col_prefix in [
            ("Before Correction", "Before_CtrlKm_"),
            ("After Correction", "Cor_CtrlKm_"),
        ]:
            col = f"{col_prefix}{k}clusters"
            predicted = central_meta[col]
            aligned = align_predictions_to_truth(predicted, truth)
            metrics = calculate_metrics(truth, aligned)
            records.append({
                "Dataset": ds["name"],
                "Target": target,
                "K": k,
                "Method": f"Central ({variant})",
                "ARI": metrics["ARI"],
                "MCC": metrics["MCC"],
                "Accuracy": metrics["Accuracy"],
                "N": metrics["N"],
            })

    return pd.DataFrame(records)


def load_precomputed_federated(dataset_name: str) -> "pd.DataFrame | None":
    """Load pre-computed federated k-means metrics if available.

    Looks for ``metrics_all.tsv`` in
    ``real_datasets/<dataset>/metrics/`` first, then falls back to
    ``real_datasets/metrics_all.tsv``.

    Returns
    -------
    pd.DataFrame or None
        Standardised metrics table, or None if no matching data found.
    """
    metrics_path = REAL_DATASETS_DIR / dataset_name / "metrics" / "metrics_all.tsv"
    if not metrics_path.exists():
        metrics_path = REAL_DATASETS_DIR / "metrics_all.tsv"
        if not metrics_path.exists():
            return None

    df = pd.read_csv(metrics_path, sep="\t")
    df = df[df["Dataset"] == dataset_name].copy()
    if df.empty:
        return None

    def _label_method(method_str: str) -> str:
        """Map raw method column values (e.g. 'proteomics_BC_Cntrl_2cls')
        to human-readable labels."""
        if "_BC_Cntrl_" in method_str:
            return "Central (Before Correction)"
        elif "_AC_Cntrl_" in method_str:
            return "Central (After Correction)"
        elif "_BC_Fed_" in method_str:
            return "Federated (Before Correction)"
        elif "_AC_Fed_" in method_str:
            return "Federated (After Correction)"
        return method_str

    df["Method"] = df["Method"].apply(_label_method)
    return df[["Dataset", "Target", "K", "Method", "ARI", "MCC", "Accuracy", "N"]]

## Plotting Functions

In [ ]:
# Color scheme: blue tones = before correction, orange tones = after correction.
# Central methods use lighter shades; federated methods use saturated shades.
METHOD_COLORS = {
    "Central (Before Correction)": "#9ecae1",
    "Central (After Correction)": "#fdae6b",
    "Federated (Before Correction)": "#3182bd",
    "Federated (After Correction)": "#e6550d",
}

METHOD_ORDER = list(METHOD_COLORS.keys())


def plot_ari_bars(df: pd.DataFrame, title: str = "") -> plt.Figure:
    """Grouped bar chart of ARI, one subplot per target (condition / batch).

    Parameters
    ----------
    df : pd.DataFrame
        Metrics table with columns: Target, Method, ARI, K.
    title : str
        Figure suptitle.

    Returns
    -------
    matplotlib Figure.
    """
    targets = df["Target"].unique()
    fig, axes = plt.subplots(1, len(targets), figsize=(6 * len(targets), 5), squeeze=False)

    for ax, target in zip(axes[0], targets):
        subset = df[df["Target"] == target].copy()
        subset["Method"] = pd.Categorical(subset["Method"], categories=METHOD_ORDER, ordered=True)
        subset = subset.sort_values("Method").dropna(subset=["Method"])

        colors = [METHOD_COLORS.get(m, "#999999") for m in subset["Method"]]
        bars = ax.bar(range(len(subset)), subset["ARI"], color=colors, edgecolor="white", width=0.7)

        # Annotate each bar with the ARI value
        for bar, val in zip(bars, subset["ARI"]):
            y_pos = val + 0.02 if val >= 0 else val - 0.06
            ax.text(
                bar.get_x() + bar.get_width() / 2, y_pos, f"{val:.3f}",
                ha="center", va="bottom" if val >= 0 else "top", fontsize=9,
            )

        ax.set_xticks(range(len(subset)))
        ax.set_xticklabels([m.replace(" (", "
(") for m in subset["Method"]], fontsize=8)
        k_val = subset["K"].iloc[0]
        ax.set_title(f"Target: {target.capitalize()} (k={k_val})", fontsize=12)
        ax.set_ylabel("ARI")
        ax.axhline(y=0, color="grey", linewidth=0.5, linestyle="--")
        ax.set_ylim(min(-0.15, df["ARI"].min() - 0.1), max(1.1, df["ARI"].max() + 0.1))

    fig.suptitle(title or "ARI: K-Means vs Batch Correction", fontsize=14, y=1.02)
    fig.tight_layout()
    return fig


def plot_pca_comparison(ds: dict) -> plt.Figure:
    """2x2 PCA scatter: (before | after) x (colored by condition | colored by batch).

    The matrix is scaled the same way federated k-means scales it
    (center + variance-scale per feature, NaN -> 0), so the PCA
    coordinates match what k-means actually sees.

    Parameters
    ----------
    ds : dict
        Output of load_dataset().

    Returns
    -------
    matplotlib Figure.
    """
    fig, axes = plt.subplots(2, 2, figsize=(14, 11))
    metadata = ds["metadata"]

    for col_idx, (label, matrix) in enumerate([
        ("Before Correction", ds["before_matrix"]),
        ("After Correction", ds["corrected_matrix"]),
    ]):
        # Scale identically to how central/federated k-means preprocesses data
        data_T = scale_like_federated(matrix)
        pca = PCA(n_components=2)
        coords = pca.fit_transform(data_T)
        var_pct = pca.explained_variance_ratio_ * 100

        pca_df = pd.DataFrame({
            "PC1": coords[:, 0], "PC2": coords[:, 1],
            "Condition": metadata["condition"].values,
            "Batch": metadata["lab"].values,
        })

        for row_idx, color_by in enumerate(["Condition", "Batch"]):
            ax = axes[row_idx, col_idx]
            groups = sorted(pca_df[color_by].unique(), key=str)
            palette = sns.color_palette("tab10", n_colors=len(groups))
            for g, c in zip(groups, palette):
                mask = pca_df[color_by] == g
                ax.scatter(
                    pca_df.loc[mask, "PC1"], pca_df.loc[mask, "PC2"],
                    c=[c], label=str(g), alpha=0.6, s=30, edgecolors="none",
                )
            ax.set_xlabel(f"PC1 ({var_pct[0]:.1f}%)")
            ax.set_ylabel(f"PC2 ({var_pct[1]:.1f}%)")
            ax.set_title(f"{label} — colored by {color_by}")
            ax.legend(fontsize=7, loc="best", title=color_by, title_fontsize=8)

    fig.suptitle(f"PCA: {ds["name"]}", fontsize=14, y=1.01)
    fig.tight_layout()
    return fig


def plot_multi_dataset_ari(all_metrics: pd.DataFrame) -> plt.Figure:
    """Grid of ARI bar charts: rows = datasets, columns = targets.

    Parameters
    ----------
    all_metrics : pd.DataFrame
        Combined metrics from all datasets (columns: Dataset, Target, Method, ARI, K).

    Returns
    -------
    matplotlib Figure.
    """
    datasets = all_metrics["Dataset"].unique()
    targets = all_metrics["Target"].unique()
    n_cols = len(targets)

    fig, axes = plt.subplots(
        len(datasets), n_cols,
        figsize=(6 * n_cols, 4 * len(datasets)),
        squeeze=False,
    )

    for row_idx, ds_name in enumerate(datasets):
        for col_idx, target in enumerate(targets):
            ax = axes[row_idx, col_idx]
            subset = all_metrics[
                (all_metrics["Dataset"] == ds_name) & (all_metrics["Target"] == target)
            ].copy()

            if subset.empty:
                ax.set_visible(False)
                continue

            subset["Method"] = pd.Categorical(
                subset["Method"], categories=METHOD_ORDER, ordered=True
            )
            subset = subset.sort_values("Method").dropna(subset=["Method"])

            colors = [METHOD_COLORS.get(m, "#999999") for m in subset["Method"]]
            bars = ax.bar(
                range(len(subset)), subset["ARI"],
                color=colors, edgecolor="white", width=0.7,
            )
            for bar, val in zip(bars, subset["ARI"]):
                y_pos = val + 0.02 if val >= 0 else val - 0.06
                ax.text(
                    bar.get_x() + bar.get_width() / 2, y_pos, f"{val:.2f}",
                    ha="center", va="bottom" if val >= 0 else "top", fontsize=8,
                )

            ax.set_xticks(range(len(subset)))
            ax.set_xticklabels(
                [m.split("(")[0].strip() for m in subset["Method"]], fontsize=8
            )
            ax.axhline(y=0, color="grey", linewidth=0.5, linestyle="--")
            k_val = subset["K"].iloc[0]
            ax.set_title(f"{ds_name} — {target} (k={k_val})", fontsize=11)
            ax.set_ylabel("ARI")

    # Shared legend at the bottom
    handles = [plt.Rectangle((0, 0), 1, 1, facecolor=c) for c in METHOD_COLORS.values()]
    fig.legend(
        handles, METHOD_COLORS.keys(), loc="lower center",
        ncol=len(METHOD_COLORS), fontsize=9, bbox_to_anchor=(0.5, -0.04),
    )
    fig.suptitle("ARI: K-Means Clustering vs Batch Correction", fontsize=14, y=1.01)
    fig.tight_layout()
    return fig

---
## Per-Dataset Evaluation

For each dataset: load data, run central k-means, compute ARI, and optionally
merge pre-computed federated k-means results.

In [ ]:
# Accumulate metrics across datasets for the multi-dataset comparison at the end.
all_central_metrics = []
all_combined_metrics = []
loaded_datasets = {}

for ds_name in DATASETS:
    print(f"
{"=" * 60}")
    print(f"  Dataset: {ds_name}")
    print(f"{"=" * 60}")

    # Load uncorrected + corrected matrices from evaluation_data/
    ds = load_dataset(ds_name, corrected_source=CORRECTED_SOURCE, remove_na=REMOVE_NA)
    loaded_datasets[ds_name] = ds

    # Run central k-means and compute metrics (ARI, MCC, accuracy)
    central_df = evaluate_dataset(ds, seed=SEED)
    all_central_metrics.append(central_df)

    # Try to load federated k-means results from real_datasets/<dataset>/metrics/
    fed_df = load_precomputed_federated(ds_name)
    if fed_df is not None:
        combined = pd.concat([central_df, fed_df], ignore_index=True)
        # Deduplicate: prefer freshly computed central over pre-computed
        combined = combined.drop_duplicates(subset=["Dataset", "Target", "Method"], keep="first")
        print("  -> Loaded pre-computed federated results")
    else:
        combined = central_df
        print("  -> No pre-computed federated results found")

    all_combined_metrics.append(combined)

central_metrics = pd.concat(all_central_metrics, ignore_index=True)
combined_metrics = pd.concat(all_combined_metrics, ignore_index=True)

print(f"
Computed metrics for {len(DATASETS)} dataset(s).")

### Metrics Table (Central K-Means)

In [ ]:
# Display central k-means metrics with ARI heatmap coloring
display_cols = ["Dataset", "Target", "K", "Method", "ARI", "MCC", "Accuracy", "N"]
central_metrics[display_cols].style.format(
    {"ARI": "{:.4f}", "MCC": "{:.4f}", "Accuracy": "{:.4f}"}
).background_gradient(subset=["ARI"], cmap="RdYlGn", vmin=-0.1, vmax=1.0)

### ARI Bar Charts (Per Dataset)

In [ ]:
# One figure per dataset showing ARI bars for each target (condition / batch)
for ds_name in DATASETS:
    ds_metrics = combined_metrics[combined_metrics["Dataset"] == ds_name]
    fig = plot_ari_bars(ds_metrics, title=f"ARI: {ds_name}")
    plt.show()

### PCA: Before vs After Correction

In [ ]:
# 2x2 PCA plots per dataset: (before | after) x (condition | batch coloring)
for ds_name in DATASETS:
    fig = plot_pca_comparison(loaded_datasets[ds_name])
    plt.show()

---
## Multi-Dataset ARI Comparison

Combined view across all evaluated datasets (one row per dataset, one column per target).

In [ ]:
if len(DATASETS) > 1:
    fig = plot_multi_dataset_ari(combined_metrics)
    plt.show()
else:
    print("Only one dataset selected — multi-dataset comparison skipped.")

### Full Metrics Table (Central + Federated)

In [ ]:
# Combined table including federated k-means results where available
combined_metrics[display_cols].style.format(
    {"ARI": "{:.4f}", "MCC": "{:.4f}", "Accuracy": "{:.4f}"}
).background_gradient(subset=["ARI"], cmap="RdYlGn", vmin=-0.1, vmax=1.0)

---
## Summary

Key ARI changes after correction. A positive delta is "good" for condition
(biology more separable) and "bad" for batch (technical signal not removed).
Vice versa for negative delta.

In [ ]:
# Build a summary table showing ARI delta per (dataset, target) pair.
summary_rows = []
for ds_name in DATASETS:
    ds_df = combined_metrics[combined_metrics["Dataset"] == ds_name]
    for target in ["condition", "batch"]:
        t_df = ds_df[ds_df["Target"] == target]
        before_central = t_df.loc[t_df["Method"] == "Central (Before Correction)", "ARI"]
        after_central = t_df.loc[t_df["Method"] == "Central (After Correction)", "ARI"]

        if not before_central.empty and not after_central.empty:
            delta = after_central.iloc[0] - before_central.iloc[0]
            summary_rows.append({
                "Dataset": ds_name,
                "Target": target,
                "ARI Before": before_central.iloc[0],
                "ARI After": after_central.iloc[0],
                "Delta ARI": delta,
                "Direction": "improved" if (
                    (target == "condition" and delta > 0)
                    or (target == "batch" and delta < 0)
                ) else "worsened" if delta != 0 else "unchanged",
            })

summary = pd.DataFrame(summary_rows)

# Highlight direction: green = improved, red = worsened
summary.style.format(
    {"ARI Before": "{:.4f}", "ARI After": "{:.4f}", "Delta ARI": "{:+.4f}"}
).map(
    lambda v: "color: green" if v == "improved" else ("color: red" if v == "worsened" else ""),
    subset=["Direction"],
)